In [1]:
%pip install keras

In [2]:
import os
import time
import json
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


Load dataset

In [3]:
DATASET_PATH = r"C:\Users\12dem\OneDrive\Desktop\Mtech_assignments\mid 2\DNN\CNN\plantvillage dataset" 

print(os.listdir(DATASET_PATH))


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\12dem\\OneDrive\\Desktop\\Mtech_assignments\\mid 2\\DNN\\CNN\\plantvillage dataset'

data loading and parting

In [8]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.1   # 90/10 split
)

train_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)

val_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

NUM_CLASSES = train_generator.num_classes
print("Number of classes:", NUM_CLASSES)


Found 2855 images belonging to 4 classes.
Found 316 images belonging to 4 classes.
Number of classes: 4


CNN

In [ ]:
custom_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),   
    layers.Dense(NUM_CLASSES, activation="softmax")
])

custom_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

custom_model.summary()


c:\Users\12dem\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,652 (76.77 KB)

 Trainable params: 19,652 (76.77 KB)

 Non-trainable params: 0 (0.00 B)

Train cnn

In [10]:
start_time = time.time()

history_custom = custom_model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator
)

custom_time = time.time() - start_time


Epoch 1/5
26/90 ━━━━━━━━━━━━━━━━━━━━ 44s 700ms/step - accuracy: 0.4680 - loss: 1.2839

KeyboardInterrupt: 

In [ ]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False   # ✅ REQUIRED

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

transfer_model = models.Model(inputs=base_model.input, outputs=outputs)

transfer_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

transfer_model.summary()


In [ ]:
start_time = time.time()

history_transfer = transfer_model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator
)

transfer_time = time.time() - start_time


In [ ]:
y_true = val_generator.classes

y_pred_custom = np.argmax(custom_model.predict(val_generator), axis=1)
y_pred_transfer = np.argmax(transfer_model.predict(val_generator), axis=1)

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro"),
        "recall": recall_score(y_true, y_pred, average="macro"),
        "f1_score": f1_score(y_true, y_pred, average="macro")
    }

custom_metrics = compute_metrics(y_true, y_pred_custom)
transfer_metrics = compute_metrics(y_true, y_pred_transfer)

print("Custom CNN Metrics:", custom_metrics)
print("Transfer Learning Metrics:", transfer_metrics)


In [ ]:
def get_assignment_results():
    return {
        "custom_cnn": {
            "metrics": custom_metrics,
            "training_time": custom_time
        },
        "transfer_learning": {
            "metrics": transfer_metrics,
            "training_time": transfer_time,
            "base_model": "ResNet50",
            "frozen_layers": len(base_model.layers),
            "trainable_layers": 2
        }
    }

print(json.dumps(get_assignment_results(), indent=4))
